In [22]:
import lmstudio as lms
from google.cloud import bigquery
import pandas as pd
import json
import time
import requests
import re
import os
import csv
import numpy as np
from pydantic import BaseModel, Field

output_csv = "carmax_specifications.csv"
error_csv = "carmax_errors.csv"

# Define columns explicitly to avoid undefined variable errors
columns = [
    'year', 'make', 'model', 'trim',
    'horsepower', 'torque', 'displacement', 'cylinders',
    'top_speed','accel', 'qmile_time', 'qmile_speed',
    'weight', 'fuel_type', 'fuel_economy',
    'quality_rating', 'reliability_rating'
]

# Create Output File if it doesn't exist
if not os.path.exists(output_csv):
    print(f"Creating {output_csv}...")
    pd.DataFrame(columns=columns).to_csv(output_csv, index=False)

# Create Error File if it doesn't exist
if not os.path.exists(error_csv):
    print(f"Creating {error_csv}...")
    pd.DataFrame(columns=["car_name", "error_message"]).to_csv(error_csv, index=False)



In [11]:
print("Reading Carmax data...\n")

df_original = pd.read_csv("carmax_USA.csv")

df_original = df_original.replace({np.nan:None})

df = df_original[['year', 'make', 'model', 'trim']].drop_duplicates()

df.head()


Reading Carmax data...



,year,make,model,trim
0,2021,Mercedes-Benz,C63 AMG,None
1,2019,Mercedes-Benz,C63 AMG,None
2,2016,Mercedes-Benz,C63 AMG,None
3,2018,Mercedes-Benz,C63 AMG,S
5,2019,Mercedes-Benz,C63 AMG,S


In [39]:
# LM Studio connection

class CarSpec(BaseModel):
    horsepower: int
    torque: int
    displacement: float
    cylinders: int
    top_speed: int
    accel: float = Field(description="0-60 mph acceleration in seconds")
    qmile_time: float = Field(description="Quarter mile time in seconds")
    qmile_speed: int = Field(description="Quarter mile trap speed in mph")
    weight: int = Field(description="Weight in kg")    
    fuel_type: str
    fuel_economy: int
    quality_rating: int
    reliability_rating: int

SERVER_API_HOST = "localhost:1234"
lms.configure_default_client(SERVER_API_HOST)

if lms.Client.is_valid_api_host(SERVER_API_HOST):
    print(f"An LM Studio API server instance is available at {SERVER_API_HOST}")
else:
    print("No LM Studio API server instance found at {SERVER_API_HOST}")

llm_model = lms.llm()


LMStudioClientError: Default client is already created, cannot set its API host.

In [24]:
# check for already processed records
processed_data = set()
existing_data = pd.read_csv(output_csv)
processed_data=set(zip(
    existing_data['year'].astype(str),
    existing_data['make'].astype(str),
    existing_data['model'].astype(str),
    existing_data['trim'].astype(str)
))

print(f"Found {len(processed_data)} already processed records\n")

keys = pd.Series(
    zip(
        df['year'].astype(str),
        df['make'].astype(str),
        df['model'].astype(str),
        df['trim'].astype(str),
    ),
    index=df.index,
)
cars_to_process = df[~keys.isin(processed_data)]


print(f"Total unique combinations: {len(df)}")
print(f"Already processed: {len(processed_data)}")
print(f"Remaining to process: {len(cars_to_process)}")

Found 0 already processed records

Total unique combinations: 686
Already processed: 0
Remaining to process: 686


In [83]:
# Step 3: Define LLM request helper functions

system_prompt = """You are a car specification assistant. Return ONLY JSON with the fields: horsepower, torque,
    engine displacement, cylinders acceleration (0-60mph), quarter mile time, quarter mile trap speed, weight, fuel_type,
    quality rating, reliability rating. Do not include explanations. Return only the JSON in format like this:
    {
        "horsepower": 469,
        "torque": 516,
        "displacement": 4.0,
        "cylinders": 8,
        "top_speed": 155,
        "accel": 4.4,
        "qmile_time": 12.8,
        "qmile_speed": 112,
        "weight": 2200,
        "fuel_type": "petrol",
        "fuel_economy": 20,
        "quality_rating": 92,
        "reliability_rating": 55
    }
    this is a 2020 Mercedes S560 as the example, replace values with the actual value"""


def create_llm_prompt(row):
    print(f"Creating prompt for {row['year']} {row['make']} {row['model']} {row['trim']}...\n\n\n")
    return f"""Provide specifications for the following vehicle:
Year: {row['year']}
Make: {row['make']}
Model: {row['model']}
Trim: {row['trim']}

Guidelines:
- Horsepower in imperial BHP, torque in lb-ft, displacement in litres.
- Acceleration (0-60mph) and quarter-mile time in seconds.
- Weight in kg.
- Quality score (1-100) with 1 being poor and 100 being perfect refined luxury (eg Rolls Royce, private jet), 
factor in material usage, sound deadening, build quality etc.
- Reliability score (1-100) with 1 being the most unreliable and 100 being able to reach 1m miles with very little maintenance.
For reference a 2020 Mercedes S560 has a quality rating of 92 and a reliability rating of 55, and a 2004 Toyota Prius has quality 40 and reliability 92.

"""
    return prompt

def send_llm_request(user_prompt):
    chat = lms.Chat(system_prompt)
    chat.add_user_message(user_prompt)

    response = llm_model.respond(
        chat,
        response_format = CarSpec
    )
    return response

def extract_json_block(response):
    """Removes thinking tags/preambles and extracts the first valid JSON object."""
    # 1. Remove <think>...</think> blocks if present
    cleaned = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
    
    # 2. Extract the outermost JSON object between the first { and last }
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if match:
        return match.group(0).strip()
    
    # Fallback to cleaned text if no curly braces matched
    return cleaned


def parse_llm_response(raw_response, row_df):
    print("Parsing data...")
    content_str = raw_response.content if hasattr(raw_response, "content") else str(raw_response)

    # 3. Parse with Pydantic
    parsed_data = CarSpec.model_validate_json(content_str)
    
    # Now .model_dump() and .model_dump_json() work as expected:
    data_dict = parsed_data.model_dump()
    
    combined_dict = {**row_df.to_dict(), **data_dict}
    combined_row_df = pd.DataFrame([combined_dict])
    print(combined_row_df)
    return combined_row_df


def append_to_csv(appending_row, filepath):
    print(f"Appending data to {filepath}...")
    file_exists = os.path.exists(filepath) and os.path.getsize(output_csv) > 0

    appending_row.to_csv(
    output_csv,
    mode="a",
    header=not file_exists,  # Only writes header if the file is new/empty
    index=False
)
    print(f"Operation Complete!\n\n\n")


In [ ]:
enriched_df_unique=

In [85]:
x=df.iloc[300]
print(x)

prompt=create_llm_prompt(x)
response=send_llm_request(prompt)
print(response)
response=extract_json_block(response)
parsed_data=parse_llm_response(response, x)
#append_to_csv(parsed_data, str(output_csv))

year         2016
make     Cadillac
model    Escalade
trim       Luxury
Name: 1851, dtype: object
Creating prompt for 2016 Cadillac Escalade Luxury...



Here's a thinking process:

1.  **Analyze User Input:**
   - Vehicle: 2016 Cadillac Escalade Luxury
   - Required JSON fields: horsepower, torque, engine displacement, cylinders, acceleration (0-60mph), quarter mile time, quarter mile trap speed, weight, fuel_type, quality rating, reliability rating
   - Constraints: Return ONLY JSON. No explanations. Specific units: BHP, lb-ft, litres, seconds, kg. Quality/Reliability scores 1-100.
   - Example format provided.

2.  **Gather Vehicle Specifications (2016 Cadillac Escalade Luxury):**
   - *Engine:* 6.2L V8 (LT1)
   - *Horsepower:* 420 hp (SAE net) -> Convert to BHP? 1 hp ≈ 1.01387 BHP. So ~425 BHP. I'll use 420 or 425. Let's stick to ~420 BHP (often listed as 420 hp, which is roughly 425 BHP). I'll use 420 for simplicity, or 425. Actually, SAE hp and BHP are very close. I'll use 420.
 

TypeError: expected string or bytes-like object, got 'PredictionResult'

In [82]:
for index, row in cars_to_process.iterrows():
    prompt=create_llm_prompt(row)
    response=send_llm_request(prompt)
    parsed_data=parse_llm_response(response, row)
    append_to_csv(parsed_data, str(output_csv))

Creating prompt for 2021 Mercedes-Benz C63 AMG None...



Parsing data...


ValidationError: 1 validation error for CarSpec
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='Here\'s a thinking proce...   "weight": 1735\n   }', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid

In [35]:
prompt=f""" Provide specifications for the following vehicle:\n
        Year: 2019
        Make: Mercedes-Benz
        Model: C63
        Trim: S
        Please return ONLY valid JSON with these exact fields:\n
        

horsepower to be given in imperial BHP, not PS, torque is lb-ft, engine displacement is in litres, cylinders is num of cylinders in the engine,
accel is 0-60mph seconds, quarter mile time is in seconds, quarter mile trap speed mph,
weight in kg,
fuel_type is petrol, diesel, hybrid, electric,
fuel economy is mpg or mpg equivalent in US mpg,
quality rating is a numeric score from 1-100 with 1 being poor and 100 being the epitome of luxury,
reliability rating is how reliable the car is on a scale from 1-100 with 1 being terrible and 100 being almost bulletproof and will last 1m miles with just routine maintenance.
For reference a 2020 Mercedes S560 has a quality rating of 92 and a reliability rating of 55, and a 2004 Toyota Prius has quality 58 and reliability 92.
        """
response = llm_model.respond(
        prompt,
        response_format = CarSpec
    )


In [51]:
content_str = response.content if hasattr(response, "content") else str(response)

# 3. Parse with Pydantic
parsed_data = CarSpec.model_validate_json(content_str)

# Now .model_dump() and .model_dump_json() work as expected:
data_dict = parsed_data.model_dump()
print(data_dict)

{'horsepower': 469, 'torque': 479, 'displacement': 2.0, 'cylinders': 4, 'top_speed': 180, 'accel': 3.9, 'qmile_time': 12.1, 'qmile_speed': 108, 'weight': 1730, 'fuel_type': 'petrol', 'fuel_economy': 24, 'quality_rating': 85, 'reliability_rating': 60}


In [70]:
print(os.path.exists(output_csv) )
print(os.path.getsize(output_csv))

True
218
